# Yelp Image Embedding Pipeline

This notebook prepares and generates image embeddings for the full New Orleans
food, drink, nightlife and hospitality discovery catalogue.

Image embeddings are generated for all eligible catalogue businesses with
available photographs. The personalisation business subset is retained only
as a reference so that the resulting embeddings can later be filtered for
personalised recommendation experiments.

## Stage D1 — Load and Verify Frozen Input Datasets

This stage reloads the approved datasets produced by the exploration notebook
and confirms that the photo metadata is correctly linked to the full business
catalogue. No image files are processed or embedded during this stage.

In [6]:
from pathlib import Path
import pandas as pd

# Current project folder
project_root = Path.cwd()

# Frozen dataset folder
data_dir = (
    project_root
    / "processed_data"
    / "new_orleans_subset"
)

required_files = {
    "full_businesses": (
        data_dir
        / "new_orleans_food_hospitality_businesses.parquet"
    ),
    "full_photos": (
        data_dir
        / "new_orleans_food_hospitality_photos.parquet"
    ),
    "personalisation_businesses": (
        data_dir
        / "new_orleans_personalisation_businesses.parquet"
    ),
    "registry": (
        data_dir
        / "dataset_registry.csv"
    )
}

missing_files = [
    str(file_path)
    for file_path in required_files.values()
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(missing_files)
        + "\n\nCurrent working directory:\n"
        + str(project_root)
    )

print("Project root:", project_root)
print("Dataset folder:", data_dir)
print("\nAll required image-pipeline inputs were found.")

Project root: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration
Dataset folder: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_subset

All required image-pipeline inputs were found.


In [7]:
catalogue_businesses = pd.read_parquet(
    required_files["full_businesses"]
)

catalogue_photos = pd.read_parquet(
    required_files["full_photos"]
)

personalisation_businesses = pd.read_parquet(
    required_files["personalisation_businesses"]
)

dataset_registry = pd.read_csv(
    required_files["registry"]
)

print(
    "Full catalogue businesses:",
    len(catalogue_businesses)
)

print(
    "Photo metadata rows:",
    len(catalogue_photos)
)

print(
    "Personalisation businesses:",
    len(personalisation_businesses)
)

Full catalogue businesses: 3505
Photo metadata rows: 19456
Personalisation businesses: 2516


In [8]:
required_photo_columns = {
    "photo_id",
    "business_id",
    "caption",
    "label"
}

missing_photo_columns = (
    required_photo_columns
    - set(catalogue_photos.columns)
)

if missing_photo_columns:
    raise ValueError(
        "Required photo columns are missing: "
        + ", ".join(sorted(missing_photo_columns))
    )

print("Required photo columns are present.")

print("\nPhoto columns:")
print(catalogue_photos.columns.tolist())

Required photo columns are present.

Photo columns:
['photo_id', 'business_id', 'caption', 'label', 'caption_clean', 'has_caption']


In [9]:
catalogue_business_ids = set(
    catalogue_businesses["business_id"]
)

personalisation_business_ids = set(
    personalisation_businesses["business_id"]
)

photo_business_ids = set(
    catalogue_photos["business_id"]
)

caption_clean = (
    catalogue_photos["caption"]
    .fillna("")
    .astype(str)
    .str.strip()
)

photo_input_audit = pd.Series({
    "Catalogue business rows":
        len(catalogue_businesses),

    "Unique catalogue businesses":
        catalogue_businesses["business_id"].nunique(),

    "Photo metadata rows":
        len(catalogue_photos),

    "Unique photo IDs":
        catalogue_photos["photo_id"].nunique(),

    "Duplicate photo IDs":
        catalogue_photos["photo_id"]
        .duplicated()
        .sum(),

    "Businesses represented by photos":
        catalogue_photos["business_id"].nunique(),

    "Photo business IDs absent from catalogue":
        len(
            photo_business_ids
            - catalogue_business_ids
        ),

    "Missing photo IDs":
        catalogue_photos["photo_id"]
        .isna()
        .sum(),

    "Missing business IDs":
        catalogue_photos["business_id"]
        .isna()
        .sum(),

    "Blank captions":
        caption_clean.eq("").sum(),

    "Captioned photos":
        caption_clean.ne("").sum(),

    "Missing labels":
        catalogue_photos["label"]
        .isna()
        .sum(),

    "Unique photo labels":
        catalogue_photos["label"]
        .nunique(),

    "Personalisation businesses":
        len(personalisation_business_ids),

    "Personalisation businesses with photos":
        catalogue_photos.loc[
            catalogue_photos["business_id"].isin(
                personalisation_business_ids
            ),
            "business_id"
        ].nunique()
})

photo_input_audit

Catalogue business rows                      3505
Unique catalogue businesses                  3505
Photo metadata rows                         19456
Unique photo IDs                            19456
Duplicate photo IDs                             0
Businesses represented by photos             1977
Photo business IDs absent from catalogue        0
Missing photo IDs                               0
Missing business IDs                            0
Blank captions                               9165
Captioned photos                            10291
Missing labels                                  0
Unique photo labels                             5
Personalisation businesses                   2516
Personalisation businesses with photos       1729
dtype: int64

In [10]:
photo_label_summary = (
    catalogue_photos["label"]
    .fillna("missing")
    .value_counts()
    .rename_axis("label")
    .to_frame("photos")
)

photo_label_summary["percentage"] = (
    photo_label_summary["photos"]
    / len(catalogue_photos)
    * 100
).round(2)

photo_label_summary

,photos,percentage
label,,
food,9321,47.91
inside,6672,34.29
outside,1915,9.84
drink,1422,7.31
menu,126,0.65


In [11]:
photo_coverage_summary = pd.DataFrame({
    "scope": [
        "Full discovery catalogue",
        "Personalisation subset"
    ],
    "total_businesses": [
        len(catalogue_business_ids),
        len(personalisation_business_ids)
    ],
    "businesses_with_photos": [
        len(
            catalogue_business_ids
            & photo_business_ids
        ),
        len(
            personalisation_business_ids
            & photo_business_ids
        )
    ]
})

photo_coverage_summary[
    "businesses_without_photos"
] = (
    photo_coverage_summary["total_businesses"]
    - photo_coverage_summary["businesses_with_photos"]
)

photo_coverage_summary[
    "photo_coverage_percentage"
] = (
    photo_coverage_summary["businesses_with_photos"]
    / photo_coverage_summary["total_businesses"]
    * 100
).round(2)

photo_coverage_summary

,scope,total_businesses,businesses_with_photos,businesses_without_photos,photo_coverage_percentage
0,Full discovery catalogue,3505,1977,1528,56.41
1,Personalisation subset,2516,1729,787,68.72


## Stage D2.1 — Locate the Physical Image Folder

The folder containing the extracted Yelp image files is identified and tested
using a sample of photo IDs from the frozen New Orleans photo metadata.

In [12]:
from pathlib import Path

# Replace this with the actual folder containing the Yelp image files
photo_dir = Path(
    "/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/Image_files/photos"
)

print("Configured photo folder:")
print(photo_dir.resolve())

print("\nFolder exists:", photo_dir.exists())
print("Is a directory:", photo_dir.is_dir())

Configured photo folder:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/Image_files/photos

Folder exists: True
Is a directory: True


In [13]:
sample_photo_ids = (
    catalogue_photos["photo_id"]
    .astype(str)
    .head(20)
    .tolist()
)

supported_extensions = [
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
]

sample_results = []

for photo_id in sample_photo_ids:
    matched_path = None

    for extension in supported_extensions:
        candidate_path = (
            photo_dir
            / f"{photo_id}{extension}"
        )

        if candidate_path.exists():
            matched_path = candidate_path
            break

    sample_results.append({
        "photo_id": photo_id,
        "file_found": matched_path is not None,
        "filename": (
            matched_path.name
            if matched_path is not None
            else None
        ),
        "extension": (
            matched_path.suffix.lower()
            if matched_path is not None
            else None
        )
    })

sample_file_check = pd.DataFrame(
    sample_results
)

sample_file_check

,photo_id,file_found,filename,extension
0,vkr8T0scuJmGVvN2HJelEA,True,vkr8T0scuJmGVvN2HJelEA.jpg,.jpg
1,mcjlyGuLFJ0t4vDixycCSg,True,mcjlyGuLFJ0t4vDixycCSg.jpg,.jpg
2,GBVORD8dr33TxT0qKwMOjQ,True,GBVORD8dr33TxT0qKwMOjQ.jpg,.jpg
3,xqkg5INxPk4_ZWxIaO8tPQ,True,xqkg5INxPk4_ZWxIaO8tPQ.jpg,.jpg
4,oG0ECMHx6OYjIE9JyLDjYA,True,oG0ECMHx6OYjIE9JyLDjYA.jpg,.jpg
5,L7fWHkgJ03_zySL1WWtmvg,True,L7fWHkgJ03_zySL1WWtmvg.jpg,.jpg
6,tBFOXD-qPtXPYNWvyeWr0A,True,tBFOXD-qPtXPYNWvyeWr0A.jpg,.jpg
7,m28V3Pvk1YW-bDkj6O6kkQ,True,m28V3Pvk1YW-bDkj6O6kkQ.jpg,.jpg
8,OPCnM6rMMis1XxF8lARJ2w,True,OPCnM6rMMis1XxF8lARJ2w.jpg,.jpg
9,SZ3NDrbN8IyamXlSQkV35g,True,SZ3NDrbN8IyamXlSQkV35g.jpg,.jpg


In [14]:
sample_location_summary = pd.Series({
    "Sample metadata records checked":
        len(sample_file_check),

    "Sample files found":
        sample_file_check["file_found"].sum(),

    "Sample files missing":
        (~sample_file_check["file_found"]).sum(),

    "All sample files found":
        sample_file_check["file_found"].all(),

    "Detected extensions":
        ", ".join(
            sorted(
                sample_file_check["extension"]
                .dropna()
                .unique()
            )
        )
})

sample_location_summary

Sample metadata records checked      20
Sample files found                   20
Sample files missing                  0
All sample files found             True
Detected extensions                .jpg
dtype: object

## Stage D2.2 — Match Photo Metadata to Physical Files

Every photo record in the frozen New Orleans catalogue is matched to its
physical image file. Missing files and duplicate physical filenames are
identified before image readability testing begins.

In [15]:
supported_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
}

physical_image_files = [
    file_path
    for file_path in photo_dir.iterdir()
    if (
        file_path.is_file()
        and file_path.suffix.lower()
        in supported_extensions
    )
]

print(
    "Supported physical image files found:",
    f"{len(physical_image_files):,}"
)

Supported physical image files found: 200,098


In [16]:
from collections import defaultdict

files_by_photo_id = defaultdict(list)

for file_path in physical_image_files:
    files_by_photo_id[file_path.stem].append(
        file_path
    )

duplicate_physical_ids = {
    photo_id: paths
    for photo_id, paths in files_by_photo_id.items()
    if len(paths) > 1
}

print(
    "Unique physical photo IDs:",
    f"{len(files_by_photo_id):,}"
)

print(
    "Physical photo IDs with multiple files:",
    len(duplicate_physical_ids)
)

Unique physical photo IDs: 200,098
Physical photo IDs with multiple files: 0


In [17]:
photo_file_inventory = (
    catalogue_photos
    .copy()
    .reset_index(drop=True)
)

def find_photo_file(photo_id):
    matching_files = files_by_photo_id.get(
        str(photo_id),
        []
    )

    if not matching_files:
        return None

    # Deterministic choice if multiple extensions exist
    return sorted(
        matching_files,
        key=lambda path: (
            path.suffix.lower(),
            path.name
        )
    )[0]


photo_file_inventory["physical_path"] = (
    photo_file_inventory["photo_id"]
    .apply(find_photo_file)
)

photo_file_inventory["file_found"] = (
    photo_file_inventory["physical_path"]
    .notna()
)

photo_file_inventory["file_extension"] = (
    photo_file_inventory["physical_path"]
    .apply(
        lambda path:
        path.suffix.lower()
        if path is not None
        else None
    )
)

# Store paths as strings for easier saving later
photo_file_inventory["file_path"] = (
    photo_file_inventory["physical_path"]
    .apply(
        lambda path:
        str(path.resolve())
        if path is not None
        else None
    )
)

photo_file_inventory = (
    photo_file_inventory
    .drop(columns="physical_path")
)

In [18]:
catalogue_photo_ids = set(
    photo_file_inventory["photo_id"]
    .astype(str)
)

physical_photo_ids = set(
    files_by_photo_id.keys()
)

missing_catalogue_photo_ids = (
    catalogue_photo_ids
    - physical_photo_ids
)

unreferenced_physical_photo_ids = (
    physical_photo_ids
    - catalogue_photo_ids
)

matched_personalisation_photos = (
    photo_file_inventory.loc[
        photo_file_inventory["business_id"].isin(
            personalisation_business_ids
        )
        & photo_file_inventory["file_found"]
    ]
)

full_file_match_audit = pd.Series({
    "Catalogue photo metadata rows":
        len(photo_file_inventory),

    "Unique catalogue photo IDs":
        photo_file_inventory["photo_id"].nunique(),

    "Physical image files indexed":
        len(physical_image_files),

    "Unique physical photo IDs":
        len(physical_photo_ids),

    "Physical IDs with multiple files":
        len(duplicate_physical_ids),

    "Catalogue files successfully matched":
        photo_file_inventory["file_found"].sum(),

    "Catalogue files missing":
        (~photo_file_inventory["file_found"]).sum(),

    "Catalogue photo IDs absent from folder":
        len(missing_catalogue_photo_ids),

    "Physical photo IDs outside catalogue subset":
        len(unreferenced_physical_photo_ids),

    "Matched catalogue businesses":
        photo_file_inventory.loc[
            photo_file_inventory["file_found"],
            "business_id"
        ].nunique(),

    "Matched personalisation photos":
        len(matched_personalisation_photos),

    "Personalisation businesses with matched photos":
        matched_personalisation_photos[
            "business_id"
        ].nunique()
})

full_file_match_audit

Catalogue photo metadata rows                      19456
Unique catalogue photo IDs                         19456
Physical image files indexed                      200098
Unique physical photo IDs                         200098
Physical IDs with multiple files                       0
Catalogue files successfully matched               19456
Catalogue files missing                                0
Catalogue photo IDs absent from folder                 0
Physical photo IDs outside catalogue subset       180642
Matched catalogue businesses                        1977
Matched personalisation photos                     18829
Personalisation businesses with matched photos      1729
dtype: int64

In [19]:
matched_extension_summary = (
    photo_file_inventory.loc[
        photo_file_inventory["file_found"],
        "file_extension"
    ]
    .value_counts()
    .rename_axis("extension")
    .to_frame("files")
)

matched_extension_summary["percentage"] = (
    matched_extension_summary["files"]
    / photo_file_inventory["file_found"].sum()
    * 100
).round(2)

matched_extension_summary

,files,percentage
extension,,
.jpg,19456,100.0


In [20]:
missing_photo_files = (
    photo_file_inventory.loc[
        ~photo_file_inventory["file_found"],
        [
            "photo_id",
            "business_id",
            "label",
            "caption"
        ]
    ]
    .copy()
)

print(
    "Missing catalogue files:",
    len(missing_photo_files)
)

missing_photo_files.head(20)

Missing catalogue files: 0


,photo_id,business_id,label,caption


## Stage D2.3 — Validate Image Readability and Integrity

Every matched catalogue image is opened and fully decoded to identify empty,
truncated, corrupted or unsupported files before image selection and embedding.
Image dimensions, format and colour mode are recorded for valid files.

In [21]:
from pathlib import Path
from PIL import Image, ImageFile, UnidentifiedImageError
from tqdm.auto import tqdm
import pandas as pd

# Do not silently accept truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = False

In [22]:
def validate_image_file(file_path):
    """
    Validate one image file and return its technical properties.

    The image is opened twice:
    1. verify() checks the file structure;
    2. load() forces full pixel decoding.
    """

    path = Path(file_path)

    result = {
        "file_size_bytes": None,
        "image_readable": False,
        "image_width": None,
        "image_height": None,
        "image_format": None,
        "image_mode": None,
        "validation_error": None
    }

    try:
        result["file_size_bytes"] = path.stat().st_size

        if result["file_size_bytes"] == 0:
            result["validation_error"] = "Empty file"
            return result

        # Check the file structure
        with Image.open(path) as image:
            image.verify()

        # Reopen and fully decode the pixels
        with Image.open(path) as image:
            image.load()

            result["image_width"] = image.width
            result["image_height"] = image.height
            result["image_format"] = image.format
            result["image_mode"] = image.mode
            result["image_readable"] = True

    except FileNotFoundError:
        result["validation_error"] = "File not found"

    except UnidentifiedImageError:
        result["validation_error"] = "Unidentified image format"

    except OSError as error:
        result["validation_error"] = (
            f"OSError: {str(error)}"
        )

    except Exception as error:
        result["validation_error"] = (
            f"{type(error).__name__}: {str(error)}"
        )

    return result

In [23]:
validation_results = []

for row in tqdm(
    photo_file_inventory.itertuples(index=False),
    total=len(photo_file_inventory),
    desc="Validating catalogue images"
):
    result = validate_image_file(
        row.file_path
    )

    result["photo_id"] = row.photo_id
    validation_results.append(result)

image_validation_df = pd.DataFrame(
    validation_results
)

image_validation_df.head()

Validating catalogue images:   0%|          | 0/19456 [00:00<?, ?it/s]

,file_size_bytes,image_readable,image_width,image_height,image_format,image_mode,validation_error,photo_id
0,40485,True,600.0,400.0,JPEG,RGB,None,vkr8T0scuJmGVvN2HJelEA
1,42820,True,600.0,337.0,JPEG,RGB,None,mcjlyGuLFJ0t4vDixycCSg
2,25800,True,351.0,400.0,JPEG,RGB,None,GBVORD8dr33TxT0qKwMOjQ
3,42373,True,300.0,400.0,JPEG,RGB,None,xqkg5INxPk4_ZWxIaO8tPQ
4,28599,True,300.0,400.0,JPEG,RGB,None,oG0ECMHx6OYjIE9JyLDjYA


In [24]:
validated_photo_inventory = (
    photo_file_inventory
    .merge(
        image_validation_df,
        on="photo_id",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Validated inventory rows:",
    len(validated_photo_inventory)
)

Validated inventory rows: 19456


In [25]:
readable_images = validated_photo_inventory.loc[
    validated_photo_inventory["image_readable"]
].copy()

unreadable_images = validated_photo_inventory.loc[
    ~validated_photo_inventory["image_readable"]
].copy()

image_integrity_summary = pd.Series({
    "Catalogue image records":
        len(validated_photo_inventory),

    "Readable images":
        validated_photo_inventory[
            "image_readable"
        ].sum(),

    "Unreadable images":
        (
            ~validated_photo_inventory[
                "image_readable"
            ]
        ).sum(),

    "Zero-byte files":
        (
            validated_photo_inventory[
                "file_size_bytes"
            ] == 0
        ).sum(),

    "Missing validation results":
        validated_photo_inventory[
            "image_readable"
        ].isna().sum(),

    "Businesses with readable images":
        readable_images[
            "business_id"
        ].nunique(),

    "Personalisation businesses with readable images":
        readable_images.loc[
            readable_images["business_id"].isin(
                personalisation_business_ids
            ),
            "business_id"
        ].nunique(),

    "Minimum image width":
        readable_images["image_width"].min(),

    "Maximum image width":
        readable_images["image_width"].max(),

    "Minimum image height":
        readable_images["image_height"].min(),

    "Maximum image height":
        readable_images["image_height"].max()
})

image_integrity_summary

Catalogue image records                            19456.0
Readable images                                    19452.0
Unreadable images                                      4.0
Zero-byte files                                        0.0
Missing validation results                             0.0
Businesses with readable images                     1977.0
Personalisation businesses with readable images     1729.0
Minimum image width                                   67.0
Maximum image width                                  600.0
Minimum image height                                  99.0
Maximum image height                                 400.0
dtype: float64

In [26]:
image_format_summary = (
    readable_images["image_format"]
    .fillna("missing")
    .value_counts()
    .rename_axis("image_format")
    .to_frame("images")
)

image_format_summary["percentage"] = (
    image_format_summary["images"]
    / len(readable_images)
    * 100
).round(2)

image_format_summary

,images,percentage
image_format,,
JPEG,19401,99.74
PNG,51,0.26


In [27]:
image_mode_summary = (
    readable_images["image_mode"]
    .fillna("missing")
    .value_counts()
    .rename_axis("image_mode")
    .to_frame("images")
)

image_mode_summary["percentage"] = (
    image_mode_summary["images"]
    / len(readable_images)
    * 100
).round(2)

image_mode_summary

,images,percentage
image_mode,,
RGB,19452,100.0


In [28]:
image_error_summary = (
    unreadable_images["validation_error"]
    .fillna("Unknown error")
    .value_counts()
    .rename_axis("error")
    .to_frame("images")
)

print(
    "Unreadable image count:",
    len(unreadable_images)
)

image_error_summary

Unreadable image count: 4


,images
error,
Unidentified image format,4


In [29]:
unreadable_images[
    [
        "photo_id",
        "business_id",
        "label",
        "file_path",
        "file_size_bytes",
        "validation_error"
    ]
].head(20)

,photo_id,business_id,label,file_path,file_size_bytes,validation_error
3669,QhATx1B1n8uf8C6siMNTfA,FxosV3rl41ZlR724nnrImA,food,/Users/preye/Downloads/Masters AI:ML/Masters P...,243,Unidentified image format
6326,K6pfRNwGodm1m1gFVQlj-Q,FxosV3rl41ZlR724nnrImA,food,/Users/preye/Downloads/Masters AI:ML/Masters P...,243,Unidentified image format
12335,l_rMdwgrvjm2PyHyXBcBTw,FxosV3rl41ZlR724nnrImA,food,/Users/preye/Downloads/Masters AI:ML/Masters P...,243,Unidentified image format
19244,iX-8Xm2G7meRHUg8qhoL1A,FxosV3rl41ZlR724nnrImA,food,/Users/preye/Downloads/Masters AI:ML/Masters P...,243,Unidentified image format


## Stage D2.4 — Save the Validated Image Inventory

The complete image-validation results are saved for reproducibility. A separate
eligible-image inventory contains only readable files that can safely proceed
to image selection and embedding.

Absolute machine-specific paths are not stored. Each image is represented by
its filename and can be reconstructed using the configured photo directory.

In [30]:
image_output_dir = (
    project_root
    / "processed_data"
    / "new_orleans_image_pipeline"
)

image_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Image-pipeline output folder:")
print(image_output_dir.resolve())

Image-pipeline output folder:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline


In [31]:
validated_photo_inventory = (
    validated_photo_inventory
    .copy()
)

# Ensure the validation flag is a proper Boolean
validated_photo_inventory["image_readable"] = (
    validated_photo_inventory["image_readable"]
    .fillna(False)
    .astype(bool)
)

# Clean caption information
validated_photo_inventory["caption_clean"] = (
    validated_photo_inventory["caption"]
    .fillna("")
    .astype(str)
    .str.strip()
)

validated_photo_inventory["caption_present"] = (
    validated_photo_inventory["caption_clean"]
    .ne("")
)

# Identify photos belonging to businesses in the
# personalisation experiment
validated_photo_inventory[
    "in_personalisation_core"
] = (
    validated_photo_inventory["business_id"]
    .isin(personalisation_business_ids)
)

# Save only a portable filename rather than an
# absolute machine-specific path
validated_photo_inventory["photo_filename"] = (
    validated_photo_inventory["file_path"]
    .apply(
        lambda path:
        Path(path).name
        if pd.notna(path)
        else None
    )
)

# Final eligibility rule for image processing
validated_photo_inventory["image_eligible"] = (
    validated_photo_inventory["file_found"]
    & validated_photo_inventory["image_readable"]
    & validated_photo_inventory[
        "file_size_bytes"
    ].fillna(0).gt(0)
)

In [32]:
columns_to_remove_from_saved_inventory = [
    "file_path"
]

validated_inventory_to_save = (
    validated_photo_inventory
    .drop(
        columns=columns_to_remove_from_saved_inventory,
        errors="ignore"
    )
    .sort_values(
        ["business_id", "photo_id"]
    )
    .reset_index(drop=True)
)

# Use nullable integers because invalid images have no dimensions
integer_columns = [
    "file_size_bytes",
    "image_width",
    "image_height"
]

for column in integer_columns:
    validated_inventory_to_save[column] = (
        pd.to_numeric(
            validated_inventory_to_save[column],
            errors="coerce"
        )
        .astype("Int64")
    )

eligible_photo_inventory = (
    validated_inventory_to_save.loc[
        validated_inventory_to_save[
            "image_eligible"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

unreadable_photo_inventory = (
    validated_inventory_to_save.loc[
        ~validated_inventory_to_save[
            "image_eligible"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Complete validated inventory:",
    len(validated_inventory_to_save)
)

print(
    "Eligible readable inventory:",
    len(eligible_photo_inventory)
)

print(
    "Excluded image records:",
    len(unreadable_photo_inventory)
)

Complete validated inventory: 19456
Eligible readable inventory: 19452
Excluded image records: 4


In [33]:
reconstructed_sample_paths = (
    eligible_photo_inventory[
        "photo_filename"
    ]
    .head(20)
    .apply(
        lambda filename:
        photo_dir / filename
    )
)

portable_path_check = pd.Series({
    "Sample reconstructed paths":
        len(reconstructed_sample_paths),

    "Sample reconstructed files found":
        reconstructed_sample_paths
        .apply(lambda path: path.exists())
        .sum(),

    "All reconstructed sample files found":
        reconstructed_sample_paths
        .apply(lambda path: path.exists())
        .all()
})

portable_path_check

Sample reconstructed paths                20
Sample reconstructed files found          20
All reconstructed sample files found    True
dtype: object

In [34]:
validated_inventory_path = (
    image_output_dir
    / "new_orleans_validated_photo_inventory.parquet"
)

eligible_inventory_path = (
    image_output_dir
    / "new_orleans_eligible_photo_inventory.parquet"
)

image_error_report_path = (
    image_output_dir
    / "new_orleans_unreadable_photo_report.csv"
)

validated_inventory_to_save.to_parquet(
    validated_inventory_path,
    index=False,
    engine="pyarrow"
)

eligible_photo_inventory.to_parquet(
    eligible_inventory_path,
    index=False,
    engine="pyarrow"
)

unreadable_photo_inventory.to_csv(
    image_error_report_path,
    index=False,
    encoding="utf-8"
)

print(
    "Saved validated inventory:",
    validated_inventory_path.resolve()
)

print(
    "Saved eligible inventory:",
    eligible_inventory_path.resolve()
)

print(
    "Saved unreadable-image report:",
    image_error_report_path.resolve()
)

Saved validated inventory: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/new_orleans_validated_photo_inventory.parquet
Saved eligible inventory: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/new_orleans_eligible_photo_inventory.parquet
Saved unreadable-image report: /Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/new_orleans_unreadable_photo_report.csv


In [35]:
saved_validated_inventory = pd.read_parquet(
    validated_inventory_path
)

saved_eligible_inventory = pd.read_parquet(
    eligible_inventory_path
)

saved_unreadable_report = pd.read_csv(
    image_error_report_path
)

inventory_save_check = pd.Series({
    "Saved validated rows":
        len(saved_validated_inventory),

    "Saved validated unique photo IDs":
        saved_validated_inventory[
            "photo_id"
        ].nunique(),

    "Saved eligible rows":
        len(saved_eligible_inventory),

    "Saved eligible unique photo IDs":
        saved_eligible_inventory[
            "photo_id"
        ].nunique(),

    "Saved excluded rows":
        len(saved_unreadable_report),

    "Eligible images marked readable":
        saved_eligible_inventory[
            "image_readable"
        ].all(),

    "Eligible images marked eligible":
        saved_eligible_inventory[
            "image_eligible"
        ].all(),

    "Eligible catalogue businesses":
        saved_eligible_inventory[
            "business_id"
        ].nunique(),

    "Eligible personalisation photos":
        saved_eligible_inventory.loc[
            saved_eligible_inventory[
                "in_personalisation_core"
            ]
        ].shape[0],

    "Eligible personalisation businesses":
        saved_eligible_inventory.loc[
            saved_eligible_inventory[
                "in_personalisation_core"
            ],
            "business_id"
        ].nunique(),

    "Absolute file path excluded":
        "file_path"
        not in saved_validated_inventory.columns,

    "Validated inventory exists":
        validated_inventory_path.exists(),

    "Eligible inventory exists":
        eligible_inventory_path.exists(),

    "Error report exists":
        image_error_report_path.exists()
})

inventory_save_check

Saved validated rows                   19456
Saved validated unique photo IDs       19456
Saved eligible rows                    19452
Saved eligible unique photo IDs        19452
Saved excluded rows                        4
Eligible images marked readable         True
Eligible images marked eligible         True
Eligible catalogue businesses           1977
Eligible personalisation photos        18825
Eligible personalisation businesses     1729
Absolute file path excluded             True
Validated inventory exists              True
Eligible inventory exists               True
Error report exists                     True
dtype: object

## Stage D3.1 — Analyse Images per Business and Label Diversity

The eligible image inventory is analysed at business level to determine how
many photographs, captions and visual labels are available for each business.
These results will guide a reproducible and label-aware image-selection rule.

In [36]:
eligible_images = pd.read_parquet(
    eligible_inventory_path
)

print(
    "Eligible image records:",
    f"{len(eligible_images):,}"
)

print(
    "Businesses with eligible images:",
    f"{eligible_images['business_id'].nunique():,}"
)

print(
    "Unique eligible photo IDs:",
    f"{eligible_images['photo_id'].nunique():,}"
)

Eligible image records: 19,452
Businesses with eligible images: 1,977
Unique eligible photo IDs: 19,452


In [37]:
business_image_summary = (
    eligible_images
    .groupby("business_id")
    .agg(
        eligible_photo_count=(
            "photo_id",
            "nunique"
        ),
        unique_photo_labels=(
            "label",
            "nunique"
        ),
        captioned_photo_count=(
            "caption_present",
            "sum"
        ),
        in_personalisation_core=(
            "in_personalisation_core",
            "max"
        )
    )
    .reset_index()
)

business_image_summary[
    "uncaptioned_photo_count"
] = (
    business_image_summary["eligible_photo_count"]
    - business_image_summary["captioned_photo_count"]
)

business_image_summary[
    "caption_coverage_percentage"
] = (
    business_image_summary["captioned_photo_count"]
    / business_image_summary["eligible_photo_count"]
    * 100
).round(2)

print(
    "Business summary rows:",
    len(business_image_summary)
)

business_image_summary.head(20)

Business summary rows: 1977


,business_id,eligible_photo_count,unique_photo_labels,captioned_photo_count,in_personalisation_core,uncaptioned_photo_count,caption_coverage_percentage
0,-0__F9fnKt8uioCKztF5Ww,10,3,6,True,4,60.00
1,-1XSzguS6XLN-V6MVZMg2A,14,2,8,True,6,57.14
2,-4x3pVUUsfWmKEilWKsOZQ,1,1,0,True,1,0.00
3,-86Z04IBKxhEQ17rCOkn8g,1,1,0,False,1,0.00
4,-A2OLubXDsMRPNN7LqohPA,1,1,1,True,0,100.00
5,-AaxZJ_I4rSFOBJbBz4SlQ,16,4,5,True,11,31.25
6,-If0ps0QhOLCYVWQWs9RYg,1,1,1,True,0,100.00
7,-Jx4CvmKL84niPDlnuMNsw,1,1,1,True,0,100.00
8,-Tskf8WK17rb3ZfeFuRSWA,66,4,33,True,33,50.00
9,-UEeCTgxud4OGWshX4x-jw,80,4,40,True,40,50.00


In [38]:
# Expected Yelp photo labels
expected_labels = [
    "food",
    "inside",
    "outside",
    "drink",
    "menu"
]

# Count each photo label per business
label_counts_by_business = (
    pd.crosstab(
        eligible_images["business_id"],
        eligible_images["label"]
    )
    .reset_index()
)

# Guarantee that all expected label columns exist
for label in expected_labels:
    if label not in label_counts_by_business.columns:
        label_counts_by_business[label] = 0

label_counts_by_business = (
    label_counts_by_business[
        ["business_id"] + expected_labels
    ]
)

# Attach label counts to the business-level summary
business_image_summary = (
    business_image_summary
    .merge(
        label_counts_by_business,
        on="business_id",
        how="left",
        validate="one_to_one"
    )
)

business_image_summary[
    ["business_id"] + expected_labels
].head()

,business_id,food,inside,outside,drink,menu
0,-0__F9fnKt8uioCKztF5Ww,1,0,2,7,0
1,-1XSzguS6XLN-V6MVZMg2A,13,1,0,0,0
2,-4x3pVUUsfWmKEilWKsOZQ,1,0,0,0,0
3,-86Z04IBKxhEQ17rCOkn8g,1,0,0,0,0
4,-A2OLubXDsMRPNN7LqohPA,1,0,0,0,0


In [39]:
image_count_distribution = (
    business_image_summary[
        "eligible_photo_count"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

image_count_distribution

count    1977.000000
mean        9.839150
std        22.441341
min         1.000000
25%         2.000000
50%         4.000000
75%        10.000000
90%        22.000000
95%        37.000000
99%        87.240000
max       528.000000
Name: eligible_photo_count, dtype: float64

In [40]:
image_count_bins = [
    0,
    1,
    2,
    5,
    10,
    20,
    50,
    float("inf")
]

image_count_labels = [
    "1 image",
    "2 images",
    "3–5 images",
    "6–10 images",
    "11–20 images",
    "21–50 images",
    "More than 50 images"
]

business_image_summary[
    "image_count_group"
] = pd.cut(
    business_image_summary[
        "eligible_photo_count"
    ],
    bins=image_count_bins,
    labels=image_count_labels,
    include_lowest=True,
    right=True
)

business_image_count_groups = (
    business_image_summary[
        "image_count_group"
    ]
    .value_counts(sort=False)
    .rename_axis("image_count_group")
    .to_frame("businesses")
)

business_image_count_groups[
    "percentage"
] = (
    business_image_count_groups["businesses"]
    / len(business_image_summary)
    * 100
).round(2)

business_image_count_groups

,businesses,percentage
image_count_group,,
1 image,458,23.17
2 images,255,12.90
3–5 images,501,25.34
6–10 images,312,15.78
11–20 images,239,12.09
21–50 images,151,7.64
More than 50 images,61,3.09


In [41]:
business_label_coverage_rows = []

for label in expected_labels:
    businesses_with_label = (
        business_image_summary[label] > 0
    ).sum()

    business_label_coverage_rows.append({
        "label": label,
        "businesses_with_label":
            businesses_with_label,
        "percentage_of_photo_businesses":
            round(
                businesses_with_label
                / len(business_image_summary)
                * 100,
                2
            ),
        "total_eligible_images":
            int(
                eligible_images.loc[
                    eligible_images["label"] == label
                ].shape[0]
            )
    })

business_label_coverage = pd.DataFrame(
    business_label_coverage_rows
)

business_label_coverage

,label,businesses_with_label,percentage_of_photo_businesses,total_eligible_images
0,food,1348,68.18,9317
1,inside,1339,67.73,6672
2,outside,719,36.37,1915
3,drink,582,29.44,1422
4,menu,96,4.86,126


In [42]:
label_diversity_summary = (
    business_image_summary[
        "unique_photo_labels"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("unique_labels")
    .to_frame("businesses")
)

label_diversity_summary[
    "percentage"
] = (
    label_diversity_summary["businesses"]
    / len(business_image_summary)
    * 100
).round(2)

label_diversity_summary

,businesses,percentage
unique_labels,,
1,787,39.81
2,553,27.97
3,389,19.68
4,216,10.93
5,32,1.62


In [43]:
business_image_summary["scope"] = (
    business_image_summary[
        "in_personalisation_core"
    ]
    .map({
        True: "Personalisation core",
        False: "General discovery only"
    })
)

scope_image_summary = (
    business_image_summary
    .groupby("scope")
    .agg(
        businesses=(
            "business_id",
            "nunique"
        ),
        total_images=(
            "eligible_photo_count",
            "sum"
        ),
        mean_images_per_business=(
            "eligible_photo_count",
            "mean"
        ),
        median_images_per_business=(
            "eligible_photo_count",
            "median"
        ),
        maximum_images_per_business=(
            "eligible_photo_count",
            "max"
        ),
        mean_unique_labels=(
            "unique_photo_labels",
            "mean"
        ),
        median_unique_labels=(
            "unique_photo_labels",
            "median"
        )
    )
    .reset_index()
)

scope_image_summary[
    "mean_images_per_business"
] = (
    scope_image_summary[
        "mean_images_per_business"
    ].round(2)
)

scope_image_summary[
    "mean_unique_labels"
] = (
    scope_image_summary[
        "mean_unique_labels"
    ].round(2)
)

scope_image_summary

,scope,businesses,total_images,mean_images_per_business,median_images_per_business,maximum_images_per_business,mean_unique_labels,median_unique_labels
0,General discovery only,248,627,2.53,2.0,25,1.29,1.0
1,Personalisation core,1729,18825,10.89,4.0,528,2.18,2.0


In [44]:
business_summary_check = pd.Series({
    "Eligible image records":
        len(eligible_images),

    "Image counts represented in business summary":
        business_image_summary[
            "eligible_photo_count"
        ].sum(),

    "Business summary rows":
        len(business_image_summary),

    "Unique businesses in eligible inventory":
        eligible_images[
            "business_id"
        ].nunique(),

    "Duplicate business summary rows":
        business_image_summary[
            "business_id"
        ].duplicated().sum(),

    "Minimum images for any represented business":
        business_image_summary[
            "eligible_photo_count"
        ].min(),

    "Maximum images for one business":
        business_image_summary[
            "eligible_photo_count"
        ].max(),

    "Minimum unique labels":
        business_image_summary[
            "unique_photo_labels"
        ].min(),

    "Maximum unique labels":
        business_image_summary[
            "unique_photo_labels"
        ].max(),

    "Image totals match":
        (
            business_image_summary[
                "eligible_photo_count"
            ].sum()
            == len(eligible_images)
        )
})

business_summary_check

Eligible image records                          19452
Image counts represented in business summary    19452
Business summary rows                            1977
Unique businesses in eligible inventory          1977
Duplicate business summary rows                     0
Minimum images for any represented business         1
Maximum images for one business                   528
Minimum unique labels                               1
Maximum unique labels                               5
Image totals match                               True
dtype: object

In [45]:
label_count_reproducibility_check = pd.Series({
    "Business rows":
        len(business_image_summary),

    "Unique businesses":
        business_image_summary[
            "business_id"
        ].nunique(),

    "Food images":
        business_image_summary["food"].sum(),

    "Inside images":
        business_image_summary["inside"].sum(),

    "Outside images":
        business_image_summary["outside"].sum(),

    "Drink images":
        business_image_summary["drink"].sum(),

    "Menu images":
        business_image_summary["menu"].sum(),

    "Total images across label columns":
        business_image_summary[
            expected_labels
        ].to_numpy().sum(),

    "Expected eligible images":
        len(eligible_images),

    "Label totals match eligible inventory":
        (
            business_image_summary[
                expected_labels
            ].to_numpy().sum()
            == len(eligible_images)
        )
})

label_count_reproducibility_check

Business rows                             1977
Unique businesses                         1977
Food images                               9317
Inside images                             6672
Outside images                            1915
Drink images                              1422
Menu images                                126
Total images across label columns        19452
Expected eligible images                 19452
Label totals match eligible inventory     True
dtype: object

## Stage D3.2 — Compare Candidate Image Caps

Candidate image-selection manifests are generated using caps of 3, 5 and 10
images per business. Selection is label-aware and deterministic so that visual
diversity is prioritised before additional images are added.

The candidates are compared using image retention, business coverage, label
preservation and caption availability before a final cap is selected.

In [46]:
selection_source = eligible_images.copy()

selection_source["label"] = (
    selection_source["label"]
    .astype(str)
    .str.strip()
    .str.lower()
)

selection_source["image_area"] = (
    pd.to_numeric(
        selection_source["image_width"],
        errors="coerce"
    ).fillna(0)
    *
    pd.to_numeric(
        selection_source["image_height"],
        errors="coerce"
    ).fillna(0)
)

selection_source["file_size_bytes"] = (
    pd.to_numeric(
        selection_source["file_size_bytes"],
        errors="coerce"
    )
    .fillna(0)
)

selection_source["caption_present"] = (
    selection_source["caption_present"]
    .fillna(False)
    .astype(bool)
)

label_priority = [
    "inside",
    "food",
    "outside",
    "drink",
    "menu"
]

unexpected_labels = (
    set(selection_source["label"].unique())
    - set(label_priority)
)

if unexpected_labels:
    raise ValueError(
        "Unexpected labels found: "
        + ", ".join(sorted(unexpected_labels))
    )

print(
    "Selection-source images:",
    f"{len(selection_source):,}"
)

print(
    "Selection-source businesses:",
    f"{selection_source['business_id'].nunique():,}"
)

print(
    "Labels:",
    selection_source["label"]
    .value_counts()
    .to_dict()
)

Selection-source images: 19,452
Selection-source businesses: 1,977
Labels: {'food': 9317, 'inside': 6672, 'outside': 1915, 'drink': 1422, 'menu': 126}


In [47]:
def select_business_images(
    business_images,
    maximum_images
):
    business_images = business_images.copy()

    label_queues = {}

    for label in label_priority:

        label_images = (
            business_images.loc[
                business_images["label"] == label
            ]
            .sort_values(
                by=[
                    "image_area",
                    "caption_present",
                    "file_size_bytes",
                    "photo_id"
                ],
                ascending=[
                    False,
                    False,
                    False,
                    True
                ]
            )
        )

        if not label_images.empty:
            label_queues[label] = (
                label_images.index.tolist()
            )

    active_labels = list(
        label_queues.keys()
    )

    selected_indices = []

    # First pass:
    # one image from each available label
    for label in active_labels:

        if len(selected_indices) >= maximum_images:
            break

        selected_indices.append(
            label_queues[label].pop(0)
        )

    # Second pass:
    # fill remaining spaces by cycling labels
    while (
        len(selected_indices) < maximum_images
        and len(selected_indices) < len(business_images)
    ):

        image_added = False

        for label in active_labels:

            if len(selected_indices) >= maximum_images:
                break

            if label_queues[label]:

                selected_indices.append(
                    label_queues[label].pop(0)
                )

                image_added = True

        if not image_added:
            break

    selected_images = (
        business_images
        .loc[selected_indices]
        .copy()
    )

    selected_images["selection_rank"] = range(
        1,
        len(selected_images) + 1
    )

    selected_images["selection_cap"] = (
        maximum_images
    )

    return selected_images

In [48]:
def build_candidate_manifest(
    image_inventory,
    maximum_images
):

    selected_groups = []

    for _, business_images in (
        image_inventory.groupby(
            "business_id",
            sort=True
        )
    ):

        selected_groups.append(
            select_business_images(
                business_images,
                maximum_images
            )
        )

    candidate_manifest = (
        pd.concat(
            selected_groups,
            ignore_index=True
        )
        .sort_values(
            [
                "business_id",
                "selection_rank"
            ]
        )
        .reset_index(drop=True)
    )

    return candidate_manifest

In [49]:
candidate_caps = [
    3,
    5,
    10
]

candidate_manifests = {}

for cap in candidate_caps:

    candidate_manifests[cap] = (
        build_candidate_manifest(
            selection_source,
            maximum_images=cap
        )
    )

    print(
        f"Cap {cap}: "
        f"{len(candidate_manifests[cap]):,} "
        "selected images"
    )

Cap 3: 4,760 selected images
Cap 5: 6,681 selected images
Cap 10: 9,748 selected images


In [50]:
candidate_comparison_rows = []

for cap, manifest in candidate_manifests.items():

    selected_business_summary = (
        manifest
        .groupby("business_id")
        .agg(
            selected_photo_count=(
                "photo_id",
                "nunique"
            ),
            selected_unique_labels=(
                "label",
                "nunique"
            ),
            selected_captioned_images=(
                "caption_present",
                "sum"
            )
        )
        .reset_index()
    )

    comparison = (
        business_image_summary[
            [
                "business_id",
                "eligible_photo_count",
                "unique_photo_labels",
                "in_personalisation_core"
            ]
        ]
        .merge(
            selected_business_summary,
            on="business_id",
            how="left",
            validate="one_to_one"
        )
    )

    comparison[
        "all_available_labels_preserved"
    ] = (
        comparison["selected_unique_labels"]
        == comparison["unique_photo_labels"]
    )

    candidate_comparison_rows.append({

        "image_cap":
            cap,

        "selected_images":
            len(manifest),

        "images_retained_percentage":
            round(
                len(manifest)
                / len(selection_source)
                * 100,
                2
            ),

        "businesses_represented":
            manifest[
                "business_id"
            ].nunique(),

        "businesses_affected_by_cap":
            (
                comparison[
                    "eligible_photo_count"
                ] > cap
            ).sum(),

        "mean_selected_images":
            round(
                comparison[
                    "selected_photo_count"
                ].mean(),
                2
            ),

        "median_selected_images":
            comparison[
                "selected_photo_count"
            ].median(),

        "maximum_selected_images":
            comparison[
                "selected_photo_count"
            ].max(),

        "mean_selected_labels":
            round(
                comparison[
                    "selected_unique_labels"
                ].mean(),
                2
            ),

        "businesses_preserving_all_labels":
            comparison[
                "all_available_labels_preserved"
            ].sum(),

        "all_label_preservation_percentage":
            round(
                comparison[
                    "all_available_labels_preserved"
                ].mean()
                * 100,
                2
            ),

        "captioned_selected_images":
            manifest[
                "caption_present"
            ].sum(),

        "captioned_selected_percentage":
            round(
                manifest[
                    "caption_present"
                ].mean()
                * 100,
                2
            )
    })

candidate_cap_comparison = pd.DataFrame(
    candidate_comparison_rows
)

candidate_cap_comparison

,image_cap,selected_images,images_retained_percentage,businesses_represented,businesses_affected_by_cap,mean_selected_images,median_selected_images,maximum_selected_images,mean_selected_labels,businesses_preserving_all_labels,all_label_preservation_percentage,captioned_selected_images,captioned_selected_percentage
0,3,4760,24.47,1977,1042,2.41,3.0,3,1.92,1729,87.46,2627,55.19
1,5,6681,34.35,1977,763,3.38,4.0,5,2.07,1977,100.00,3656,54.72
2,10,9748,50.11,1977,451,4.93,4.0,10,2.07,1977,100.00,5292,54.29


In [51]:
candidate_label_rows = []

for cap, manifest in candidate_manifests.items():

    selected_label_counts = (
        manifest["label"]
        .value_counts()
    )

    for label in label_priority:

        available_count = (
            selection_source[
                "label"
            ]
            .eq(label)
            .sum()
        )

        selected_count = int(
            selected_label_counts.get(
                label,
                0
            )
        )

        candidate_label_rows.append({

            "image_cap":
                cap,

            "label":
                label,

            "selected_images":
                selected_count,

            "available_images":
                available_count,

            "label_images_retained_percentage":
                round(
                    selected_count
                    / available_count
                    * 100,
                    2
                )
                if available_count
                else 0
        })

candidate_label_comparison = pd.DataFrame(
    candidate_label_rows
)

candidate_label_comparison

,image_cap,label,selected_images,available_images,label_images_retained_percentage
0,3,inside,1773,6672,26.57
1,3,food,1824,9317,19.58
2,3,outside,753,1915,39.32
3,3,drink,376,1422,26.44
4,3,menu,34,126,26.98
5,5,inside,2484,6672,37.23
6,5,food,2583,9317,27.72
7,5,outside,855,1915,44.65
8,5,drink,662,1422,46.55
9,5,menu,97,126,76.98


In [52]:
candidate_scope_rows = []

for cap, manifest in candidate_manifests.items():

    scope_manifest = manifest.copy()

    scope_manifest["scope"] = (
        scope_manifest[
            "in_personalisation_core"
        ]
        .map({
            True:
                "Personalisation core",
            False:
                "General discovery only"
        })
    )

    scope_summary = (
        scope_manifest
        .groupby("scope")
        .agg(
            selected_images=(
                "photo_id",
                "nunique"
            ),
            represented_businesses=(
                "business_id",
                "nunique"
            )
        )
        .reset_index()
    )

    scope_summary[
        "mean_selected_images_per_business"
    ] = (
        scope_summary[
            "selected_images"
        ]
        / scope_summary[
            "represented_businesses"
        ]
    ).round(2)

    scope_summary["image_cap"] = cap

    candidate_scope_rows.append(
        scope_summary
    )

candidate_scope_comparison = (
    pd.concat(
        candidate_scope_rows,
        ignore_index=True
    )
    [
        [
            "image_cap",
            "scope",
            "selected_images",
            "represented_businesses",
            "mean_selected_images_per_business"
        ]
    ]
)

candidate_scope_comparison

,image_cap,scope,selected_images,represented_businesses,mean_selected_images_per_business
0,3,General discovery only,474,248,1.91
1,3,Personalisation core,4286,1729,2.48
2,5,General discovery only,559,248,2.25
3,5,Personalisation core,6122,1729,3.54
4,10,General discovery only,598,248,2.41
5,10,Personalisation core,9150,1729,5.29


In [53]:
candidate_integrity_rows = []

for cap, manifest in candidate_manifests.items():

    per_business_counts = (
        manifest
        .groupby("business_id")
        ["photo_id"]
        .nunique()
    )

    candidate_integrity_rows.append({

        "image_cap":
            cap,

        "selected_rows":
            len(manifest),

        "unique_selected_photo_ids":
            manifest[
                "photo_id"
            ].nunique(),

        "duplicate_selected_photo_ids":
            manifest[
                "photo_id"
            ].duplicated().sum(),

        "represented_businesses":
            manifest[
                "business_id"
            ].nunique(),

        "maximum_images_for_one_business":
            per_business_counts.max(),

        "businesses_exceeding_cap":
            (
                per_business_counts > cap
            ).sum(),

        "all_selected_images_are_eligible":
            manifest[
                "image_eligible"
            ].all()
    })

candidate_manifest_integrity = pd.DataFrame(
    candidate_integrity_rows
)

candidate_manifest_integrity

,image_cap,selected_rows,unique_selected_photo_ids,duplicate_selected_photo_ids,represented_businesses,maximum_images_for_one_business,businesses_exceeding_cap,all_selected_images_are_eligible
0,3,4760,4760,0,1977,3,0,True
1,5,6681,6681,0,1977,5,0,True
2,10,9748,9748,0,1977,10,0,True


In [54]:
print("\n=== CANDIDATE CAP COMPARISON ===")
print(
    candidate_cap_comparison
    .to_string(index=False)
)

print("\n=== LABEL COMPARISON ===")
print(
    candidate_label_comparison
    .to_string(index=False)
)

print("\n=== SCOPE COMPARISON ===")
print(
    candidate_scope_comparison
    .to_string(index=False)
)

print("\n=== MANIFEST INTEGRITY ===")
print(
    candidate_manifest_integrity
    .to_string(index=False)
)


=== CANDIDATE CAP COMPARISON ===
 image_cap  selected_images  images_retained_percentage  businesses_represented  businesses_affected_by_cap  mean_selected_images  median_selected_images  maximum_selected_images  mean_selected_labels  businesses_preserving_all_labels  all_label_preservation_percentage  captioned_selected_images  captioned_selected_percentage
         3             4760                       24.47                    1977                        1042                  2.41                     3.0                        3                  1.92                              1729                              87.46                       2627                          55.19
         5             6681                       34.35                    1977                         763                  3.38                     4.0                        5                  2.07                              1977                             100.00                       3656              

## Stage D3.3 — Finalise the Image Embedding Manifest

A maximum of five images per business is adopted following comparison with
three- and ten-image alternatives. The five-image strategy preserves every
available visual label for all 1,977 image-supported businesses while avoiding
the additional within-label redundancy introduced by the ten-image cap.

The final manifest therefore contains a deterministic, label-aware subset of
eligible photographs for image embedding.

In [55]:
final_image_manifest = (
    candidate_manifests[5]
    .copy()
    .sort_values(
        [
            "business_id",
            "selection_rank"
        ]
    )
    .reset_index(drop=True)
)

FINAL_IMAGE_CAP = 5

print(
    "Final image cap:",
    FINAL_IMAGE_CAP
)

print(
    "Final selected images:",
    f"{len(final_image_manifest):,}"
)

print(
    "Represented businesses:",
    f"{final_image_manifest['business_id'].nunique():,}"
)

Final image cap: 5
Final selected images: 6,681
Represented businesses: 1,977


In [56]:
final_image_manifest[
    "selection_strategy"
] = "label_aware_cap5"

final_image_manifest[
    "selection_version"
] = "v1"

final_image_manifest[
    "selected_for_embedding"
] = True

In [57]:
final_selected_label_counts = (
    final_image_manifest
    .groupby("business_id")["label"]
    .nunique()
    .rename("selected_unique_labels")
    .reset_index()
)

final_label_check = (
    business_image_summary[
        [
            "business_id",
            "unique_photo_labels"
        ]
    ]
    .merge(
        final_selected_label_counts,
        on="business_id",
        how="left",
        validate="one_to_one"
    )
)

final_label_check[
    "all_available_labels_preserved"
] = (
    final_label_check[
        "unique_photo_labels"
    ]
    ==
    final_label_check[
        "selected_unique_labels"
    ]
)

final_label_check_summary = pd.Series({
    "Businesses checked":
        len(final_label_check),

    "Businesses preserving all available labels":
        final_label_check[
            "all_available_labels_preserved"
        ].sum(),

    "Businesses losing an available label":
        (
            ~final_label_check[
                "all_available_labels_preserved"
            ]
        ).sum(),

    "All businesses preserve available labels":
        final_label_check[
            "all_available_labels_preserved"
        ].all()
})

final_label_check_summary

Businesses checked                            1977
Businesses preserving all available labels    1977
Businesses losing an available label             0
All businesses preserve available labels      True
dtype: object

In [58]:
final_manifest_check = pd.Series({
    "Selected image rows":
        len(final_image_manifest),

    "Unique selected photo IDs":
        final_image_manifest[
            "photo_id"
        ].nunique(),

    "Duplicate selected photo IDs":
        final_image_manifest[
            "photo_id"
        ].duplicated().sum(),

    "Represented businesses":
        final_image_manifest[
            "business_id"
        ].nunique(),

    "Maximum images per business":
        final_image_manifest
        .groupby("business_id")[
            "photo_id"
        ]
        .nunique()
        .max(),

    "Businesses exceeding cap":
        (
            final_image_manifest
            .groupby("business_id")[
                "photo_id"
            ]
            .nunique()
            > FINAL_IMAGE_CAP
        ).sum(),

    "All images eligible":
        final_image_manifest[
            "image_eligible"
        ].all(),

    "Personalisation selected images":
        final_image_manifest.loc[
            final_image_manifest[
                "in_personalisation_core"
            ]
        ].shape[0],

    "Personalisation businesses represented":
        final_image_manifest.loc[
            final_image_manifest[
                "in_personalisation_core"
            ],
            "business_id"
        ].nunique(),

    "General-only selected images":
        final_image_manifest.loc[
            ~final_image_manifest[
                "in_personalisation_core"
            ]
        ].shape[0],

    "General-only businesses represented":
        final_image_manifest.loc[
            ~final_image_manifest[
                "in_personalisation_core"
            ],
            "business_id"
        ].nunique()
})

final_manifest_check

Selected image rows                       6681
Unique selected photo IDs                 6681
Duplicate selected photo IDs                 0
Represented businesses                    1977
Maximum images per business                  5
Businesses exceeding cap                     0
All images eligible                       True
Personalisation selected images           6122
Personalisation businesses represented    1729
General-only selected images               559
General-only businesses represented        248
dtype: object

In [59]:
final_manifest_path = (
    image_output_dir
    / "new_orleans_image_embedding_manifest.parquet"
)

final_image_manifest.to_parquet(
    final_manifest_path,
    index=False,
    engine="pyarrow"
)

print(
    "Final image manifest saved to:"
)

print(
    final_manifest_path.resolve()
)

Final image manifest saved to:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/new_orleans_image_embedding_manifest.parquet


In [60]:
cap_comparison_path = (
    image_output_dir
    / "image_cap_comparison.csv"
)

label_comparison_path = (
    image_output_dir
    / "image_cap_label_comparison.csv"
)

scope_comparison_path = (
    image_output_dir
    / "image_cap_scope_comparison.csv"
)

candidate_cap_comparison.to_csv(
    cap_comparison_path,
    index=False
)

candidate_label_comparison.to_csv(
    label_comparison_path,
    index=False
)

candidate_scope_comparison.to_csv(
    scope_comparison_path,
    index=False
)

print("Saved image-cap comparison evidence.")

Saved image-cap comparison evidence.


In [61]:
saved_final_image_manifest = pd.read_parquet(
    final_manifest_path
)

final_manifest_save_check = pd.Series({
    "Saved manifest rows":
        len(saved_final_image_manifest),

    "Saved unique photo IDs":
        saved_final_image_manifest[
            "photo_id"
        ].nunique(),

    "Saved businesses":
        saved_final_image_manifest[
            "business_id"
        ].nunique(),

    "Saved maximum images per business":
        saved_final_image_manifest
        .groupby("business_id")[
            "photo_id"
        ]
        .nunique()
        .max(),

    "Saved photo IDs exactly match in-memory manifest":
        set(
            saved_final_image_manifest[
                "photo_id"
            ]
        )
        ==
        set(
            final_image_manifest[
                "photo_id"
            ]
        ),

    "Manifest file exists":
        final_manifest_path.exists(),

    "Cap comparison exists":
        cap_comparison_path.exists(),

    "Label comparison exists":
        label_comparison_path.exists(),

    "Scope comparison exists":
        scope_comparison_path.exists()
})

final_manifest_save_check

Saved manifest rows                                 6681
Saved unique photo IDs                              6681
Saved businesses                                    1977
Saved maximum images per business                      5
Saved photo IDs exactly match in-memory manifest    True
Manifest file exists                                True
Cap comparison exists                               True
Label comparison exists                             True
Scope comparison exists                             True
dtype: object

## Stage D4.1 — Select and Verify the Image Embedding Model

The pretrained CLIP ViT-B/32 model is used as the image encoder. CLIP provides
semantic image representations aligned with natural-language concepts, making
it suitable for both visual business representation and later multimodal
retrieval.

The pretrained encoder is kept frozen. This stage verifies the computational
device, preprocessing pipeline and embedding output using a single selected
image before batch embedding begins.

In [62]:
import sys
import torch
import transformers

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if hasattr(torch.backends, "mps"):
    print(
        "MPS built:",
        torch.backends.mps.is_built()
    )
    
    print(
        "MPS available:",
        torch.backends.mps.is_available()
    )

Python version: 3.10.19
PyTorch version: 2.11.0
Transformers version: 5.8.1

CUDA available: False
MPS built: True
MPS available: True


In [63]:
if torch.cuda.is_available():
    device = torch.device("cuda")

elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")

else:
    device = torch.device("cpu")

print("Selected device:", device)

Selected device: mps


In [64]:
from transformers import (
    CLIPModel,
    CLIPProcessor
)

CLIP_MODEL_NAME = (
    "openai/clip-vit-base-patch32"
)

print(
    "Loading model:",
    CLIP_MODEL_NAME
)

clip_processor = (
    CLIPProcessor.from_pretrained(
        CLIP_MODEL_NAME
    )
)

clip_model = (
    CLIPModel.from_pretrained(
        CLIP_MODEL_NAME
    )
)

clip_model = clip_model.to(device)

clip_model.eval()

# Freeze all parameters
for parameter in clip_model.parameters():
    parameter.requires_grad = False

print("Model loaded successfully.")
print("Device:", device)

Loading model: openai/clip-vit-base-patch32


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded successfully.
Device: mps


In [65]:
model_configuration = pd.Series({
    "Model":
        CLIP_MODEL_NAME,

    "Projection dimension":
        clip_model.config.projection_dim,

    "Vision hidden size":
        clip_model.config.vision_config.hidden_size,

    "Image input size":
        clip_model.config.vision_config.image_size,

    "Patch size":
        clip_model.config.vision_config.patch_size,

    "Vision layers":
        clip_model.config.vision_config.num_hidden_layers,

    "Vision attention heads":
        clip_model.config.vision_config.num_attention_heads,

    "Model training mode":
        clip_model.training,

    "Trainable parameters":
        sum(
            parameter.numel()
            for parameter in clip_model.parameters()
            if parameter.requires_grad
        ),

    "Total parameters":
        sum(
            parameter.numel()
            for parameter in clip_model.parameters()
        ),

    "Device":
        str(device)
})

model_configuration

Model                     openai/clip-vit-base-patch32
Projection dimension                               512
Vision hidden size                                 768
Image input size                                   224
Patch size                                          32
Vision layers                                       12
Vision attention heads                              12
Model training mode                              False
Trainable parameters                                 0
Total parameters                             151277313
Device                                             mps
dtype: object

In [66]:
from PIL import Image

sample_manifest_row = (
    saved_final_image_manifest.iloc[0]
)

sample_image_path = (
    photo_dir
    / sample_manifest_row["photo_filename"]
)

print(
    "Sample photo ID:",
    sample_manifest_row["photo_id"]
)

print(
    "Business ID:",
    sample_manifest_row["business_id"]
)

print(
    "Label:",
    sample_manifest_row["label"]
)

print(
    "Physical file exists:",
    sample_image_path.exists()
)

print(
    "File:",
    sample_image_path.name
)

Sample photo ID: 1Hk9SCh7PXionemIjLvABw
Business ID: -0__F9fnKt8uioCKztF5Ww
Label: food
Physical file exists: True
File: 1Hk9SCh7PXionemIjLvABw.jpg


In [67]:
with Image.open(sample_image_path) as image:
    
    sample_rgb_image = (
        image.convert("RGB")
    )

    sample_inputs = (
        clip_processor(
            images=sample_rgb_image,
            return_tensors="pt"
        )
    )

sample_pixel_values = (
    sample_inputs["pixel_values"]
    .to(device)
)

print(
    "Processed tensor shape:",
    tuple(sample_pixel_values.shape)
)

print(
    "Tensor dtype:",
    sample_pixel_values.dtype
)

print(
    "Tensor device:",
    sample_pixel_values.device
)

Processed tensor shape: (1, 3, 224, 224)
Tensor dtype: torch.float32
Tensor device: mps:0


In [68]:
with torch.inference_mode():

    sample_image_output = (
        clip_model.get_image_features(
            pixel_values=sample_pixel_values
        )
    )

# Current Transformers versions return
# BaseModelOutputWithPooling.
# The projected CLIP image embedding is stored
# in pooler_output.
if hasattr(
    sample_image_output,
    "pooler_output"
):
    sample_embedding = (
        sample_image_output.pooler_output
    )
else:
    # Compatibility with Transformers versions
    # that return the tensor directly
    sample_embedding = sample_image_output


print(
    "Returned object type:",
    type(sample_image_output).__name__
)

print(
    "Raw embedding shape:",
    tuple(sample_embedding.shape)
)

print(
    "Contains NaN:",
    torch.isnan(
        sample_embedding
    ).any().item()
)

print(
    "Contains infinity:",
    torch.isinf(
        sample_embedding
    ).any().item()
)

Returned object type: BaseModelOutputWithPooling
Raw embedding shape: (1, 512)
Contains NaN: False
Contains infinity: False


In [69]:
sample_embedding_normalised = (
    sample_embedding
    / sample_embedding.norm(
        p=2,
        dim=-1,
        keepdim=True
    )
)

sample_embedding_norm = (
    sample_embedding_normalised
    .norm(
        p=2,
        dim=-1
    )
    .item()
)

print(
    "Normalised embedding shape:",
    tuple(
        sample_embedding_normalised.shape
    )
)

print(
    "L2 norm:",
    sample_embedding_norm
)

Normalised embedding shape: (1, 512)
L2 norm: 1.0


In [70]:
sample_embedding_check = pd.Series({
    "Image exists":
        sample_image_path.exists(),

    "Input batch size":
        sample_pixel_values.shape[0],

    "Input channels":
        sample_pixel_values.shape[1],

    "Processed image height":
        sample_pixel_values.shape[2],

    "Processed image width":
        sample_pixel_values.shape[3],

    "Embedding batch size":
        sample_embedding_normalised.shape[0],

    "Embedding dimension":
        sample_embedding_normalised.shape[1],

    "Expected embedding dimension":
        clip_model.config.projection_dim,

    "Embedding dimension correct":
        (
            sample_embedding_normalised.shape[1]
            ==
            clip_model.config.projection_dim
        ),

    "Contains NaN":
        torch.isnan(
            sample_embedding_normalised
        ).any().item(),

    "Contains infinity":
        torch.isinf(
            sample_embedding_normalised
        ).any().item(),

    "Embedding norm":
        round(
            sample_embedding_norm,
            6
        ),

    "Model in evaluation mode":
        not clip_model.training,

    "Trainable parameters":
        sum(
            parameter.numel()
            for parameter
            in clip_model.parameters()
            if parameter.requires_grad
        )
})

sample_embedding_check

Image exists                     True
Input batch size                    1
Input channels                      3
Processed image height            224
Processed image width             224
Embedding batch size                1
Embedding dimension               512
Expected embedding dimension      512
Embedding dimension correct      True
Contains NaN                    False
Contains infinity               False
Embedding norm                    1.0
Model in evaluation mode         True
Trainable parameters                0
dtype: object

## Stage D4.3 — Production CLIP Image Embedding

The final 6,681-image manifest is encoded using the frozen CLIP ViT-B/32
visual encoder. Image-level embeddings are L2-normalised and saved in
restart-safe checkpoint chunks.

Checkpointing allows interrupted runs to resume without recomputing completed
images. Once all images have been processed, checkpoint chunks are consolidated
into a final embedding matrix and aligned metadata index.

In [71]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import time
from PIL import Image
from tqdm.auto import tqdm

# --------------------------------------------------
# Production configuration
# --------------------------------------------------

CLIP_EMBEDDING_DIM = 512

# Conservative defaults for each device
if device.type == "cuda":
    PRODUCTION_BATCH_SIZE = 32

elif device.type == "mps":
    PRODUCTION_BATCH_SIZE = 16

else:
    PRODUCTION_BATCH_SIZE = 8


# Save progress approximately every 256 images
CHECKPOINT_SIZE = 256


# --------------------------------------------------
# Output folders
# --------------------------------------------------

clip_output_dir = (
    image_output_dir
    / "clip_vit_b32_embeddings"
)

clip_checkpoint_dir = (
    clip_output_dir
    / "checkpoints"
)

clip_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

clip_checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------
# Production manifest
# --------------------------------------------------

production_manifest = (
    saved_final_image_manifest
    .copy()
    .reset_index(drop=True)
)

production_manifest[
    "manifest_position"
] = np.arange(
    len(production_manifest)
)


print("CLIP production configuration")
print("--------------------------------")
print(
    "Images to embed:",
    f"{len(production_manifest):,}"
)
print(
    "Businesses:",
    f"{production_manifest['business_id'].nunique():,}"
)
print(
    "Embedding dimension:",
    CLIP_EMBEDDING_DIM
)
print(
    "Device:",
    device
)
print(
    "Batch size:",
    PRODUCTION_BATCH_SIZE
)
print(
    "Checkpoint size:",
    CHECKPOINT_SIZE
)
print(
    "Checkpoint folder:"
)
print(
    clip_checkpoint_dir.resolve()
)

CLIP production configuration
--------------------------------
Images to embed: 6,681
Businesses: 1,977
Embedding dimension: 512
Device: mps
Batch size: 16
Checkpoint size: 256
Checkpoint folder:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/clip_vit_b32_embeddings/checkpoints


In [72]:
def extract_clip_image_features(
    model,
    pixel_values
):
    """
    Extract projected CLIP image features.

    Compatible with Transformers versions returning either:
    - a Tensor, or
    - BaseModelOutputWithPooling.
    """

    output = model.get_image_features(
        pixel_values=pixel_values
    )

    if hasattr(
        output,
        "pooler_output"
    ):
        features = output.pooler_output

    else:
        features = output

    return features

In [73]:
checkpoint_metadata_files = sorted(
    clip_checkpoint_dir.glob(
        "checkpoint_*.parquet"
    )
)

processed_photo_ids = set()
completed_checkpoint_rows = 0

for metadata_path in checkpoint_metadata_files:

    embedding_path = (
        metadata_path
        .with_suffix(".npy")
    )

    if not embedding_path.exists():
        raise FileNotFoundError(
            f"Checkpoint metadata exists but "
            f"embedding file is missing:\n"
            f"{embedding_path}"
        )

    checkpoint_metadata = (
        pd.read_parquet(
            metadata_path
        )
    )

    checkpoint_embeddings = (
        np.load(
            embedding_path,
            mmap_mode="r"
        )
    )

    if (
        len(checkpoint_metadata)
        != checkpoint_embeddings.shape[0]
    ):
        raise ValueError(
            f"Checkpoint row mismatch: "
            f"{metadata_path.name}"
        )

    if (
        checkpoint_embeddings.shape[1]
        != CLIP_EMBEDDING_DIM
    ):
        raise ValueError(
            f"Unexpected embedding dimension in "
            f"{embedding_path.name}: "
            f"{checkpoint_embeddings.shape}"
        )

    processed_photo_ids.update(
        checkpoint_metadata[
            "photo_id"
        ].tolist()
    )

    completed_checkpoint_rows += (
        len(checkpoint_metadata)
    )


remaining_manifest = (
    production_manifest.loc[
        ~production_manifest[
            "photo_id"
        ].isin(
            processed_photo_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "Existing checkpoint files:",
    len(checkpoint_metadata_files)
)

print(
    "Previously embedded images:",
    f"{len(processed_photo_ids):,}"
)

print(
    "Images remaining:",
    f"{len(remaining_manifest):,}"
)

Existing checkpoint files: 27
Previously embedded images: 6,681
Images remaining: 0


In [74]:
def embed_manifest_subset(
    manifest_subset,
    model,
    processor,
    photo_directory,
    device,
    batch_size
):
    """
    Embed a manifest subset using CLIP and return
    an L2-normalised float32 matrix.
    """

    embedding_batches = []

    for start_index in range(
        0,
        len(manifest_subset),
        batch_size
    ):

        batch_manifest = (
            manifest_subset.iloc[
                start_index:
                start_index + batch_size
            ]
        )

        batch_images = []

        for filename in batch_manifest[
            "photo_filename"
        ]:

            image_path = (
                photo_directory
                / filename
            )

            if not image_path.exists():
                raise FileNotFoundError(
                    f"Image disappeared after validation: "
                    f"{image_path}"
                )

            with Image.open(
                image_path
            ) as image:

                batch_images.append(
                    image
                    .convert("RGB")
                    .copy()
                )


        batch_inputs = processor(
            images=batch_images,
            return_tensors="pt"
        )

        batch_pixel_values = (
            batch_inputs[
                "pixel_values"
            ]
            .to(device)
        )


        with torch.inference_mode():

            batch_features = (
                extract_clip_image_features(
                    model=model,
                    pixel_values=(
                        batch_pixel_values
                    )
                )
            )

            # L2 normalisation
            batch_norms = (
                batch_features.norm(
                    p=2,
                    dim=-1,
                    keepdim=True
                )
            )

            if (
                batch_norms <= 0
            ).any():
                raise ValueError(
                    "Zero-norm CLIP embedding "
                    "detected."
                )

            batch_features = (
                batch_features
                / batch_norms
            )


        batch_array = (
            batch_features
            .detach()
            .cpu()
            .float()
            .numpy()
        )


        # Immediate numerical check
        if np.isnan(
            batch_array
        ).any():
            raise ValueError(
                "NaN detected in CLIP embeddings."
            )

        if np.isinf(
            batch_array
        ).any():
            raise ValueError(
                "Infinity detected in "
                "CLIP embeddings."
            )


        embedding_batches.append(
            batch_array
        )


        # Release temporary tensors
        del batch_pixel_values
        del batch_features

        if device.type == "mps":
            torch.mps.empty_cache()

        elif device.type == "cuda":
            torch.cuda.empty_cache()


    return np.vstack(
        embedding_batches
    ).astype(
        np.float32
    )

In [75]:
# --------------------------------------------------
# Determine next checkpoint number
# --------------------------------------------------

existing_checkpoint_numbers = []

for metadata_path in checkpoint_metadata_files:

    checkpoint_number = int(
        metadata_path
        .stem
        .split("_")[-1]
    )

    existing_checkpoint_numbers.append(
        checkpoint_number
    )


next_checkpoint_number = (
    max(existing_checkpoint_numbers) + 1
    if existing_checkpoint_numbers
    else 0
)


# --------------------------------------------------
# Production run
# --------------------------------------------------

production_start_time = (
    time.perf_counter()
)

newly_processed_images = 0


for chunk_start in tqdm(
    range(
        0,
        len(remaining_manifest),
        CHECKPOINT_SIZE
    ),
    desc="CLIP production checkpoints"
):

    chunk_manifest = (
        remaining_manifest.iloc[
            chunk_start:
            chunk_start + CHECKPOINT_SIZE
        ]
        .copy()
        .reset_index(drop=True)
    )

    checkpoint_number = (
        next_checkpoint_number
    )

    next_checkpoint_number += 1


    print(
        f"\nEmbedding checkpoint "
        f"{checkpoint_number:04d}"
    )

    print(
        "Images in checkpoint:",
        len(chunk_manifest)
    )


    # ----------------------------------------------
    # Generate embeddings
    # ----------------------------------------------

    chunk_embeddings = (
        embed_manifest_subset(
            manifest_subset=(
                chunk_manifest
            ),
            model=clip_model,
            processor=clip_processor,
            photo_directory=photo_dir,
            device=device,
            batch_size=(
                PRODUCTION_BATCH_SIZE
            )
        )
    )


    # ----------------------------------------------
    # Verify checkpoint before saving
    # ----------------------------------------------

    if (
        chunk_embeddings.shape
        != (
            len(chunk_manifest),
            CLIP_EMBEDDING_DIM
        )
    ):
        raise ValueError(
            "Unexpected checkpoint "
            f"embedding shape: "
            f"{chunk_embeddings.shape}"
        )


    chunk_norms = np.linalg.norm(
        chunk_embeddings,
        axis=1
    )

    if not np.allclose(
        chunk_norms,
        1.0,
        atol=1e-5
    ):
        raise ValueError(
            "Checkpoint contains "
            "non-normalised embeddings."
        )


    # ----------------------------------------------
    # Checkpoint paths
    # ----------------------------------------------

    checkpoint_stem = (
        f"checkpoint_"
        f"{checkpoint_number:04d}"
    )

    checkpoint_embedding_path = (
        clip_checkpoint_dir
        / f"{checkpoint_stem}.npy"
    )

    checkpoint_metadata_path = (
        clip_checkpoint_dir
        / f"{checkpoint_stem}.parquet"
    )


    # ----------------------------------------------
    # Atomic embedding save
    # ----------------------------------------------

    temporary_embedding_path = (
        clip_checkpoint_dir
        / f"{checkpoint_stem}.tmp"
    )

    with open(
        temporary_embedding_path,
        "wb"
    ) as file_handle:

        np.save(
            file_handle,
            chunk_embeddings
        )

    temporary_embedding_path.replace(
        checkpoint_embedding_path
    )


    # ----------------------------------------------
    # Save matching metadata
    # ----------------------------------------------

    chunk_metadata_to_save = (
        chunk_manifest[
            [
                "manifest_position",
                "photo_id",
                "business_id",
                "photo_filename",
                "label",
                "selection_rank",
                "in_personalisation_core"
            ]
        ]
        .copy()
    )

    temporary_metadata_path = (
        clip_checkpoint_dir
        / f"{checkpoint_stem}.tmp.parquet"
    )

    chunk_metadata_to_save.to_parquet(
        temporary_metadata_path,
        index=False,
        engine="pyarrow"
    )

    temporary_metadata_path.replace(
        checkpoint_metadata_path
    )


    newly_processed_images += (
        len(chunk_manifest)
    )


    elapsed = (
        time.perf_counter()
        - production_start_time
    )

    total_completed = (
        len(processed_photo_ids)
        + newly_processed_images
    )


    print(
        f"Saved: "
        f"{total_completed:,} / "
        f"{len(production_manifest):,}"
    )

    print(
        "Current checkpoint shape:",
        chunk_embeddings.shape
    )

    print(
        "Elapsed minutes:",
        round(
            elapsed / 60,
            2
        )
    )


print(
    "\nProduction checkpointing run complete."
)

CLIP production checkpoints: 0it [00:00, ?it/s]


Production checkpointing run complete.


In [76]:
final_checkpoint_metadata_files = sorted(
    clip_checkpoint_dir.glob(
        "checkpoint_*.parquet"
    )
)

all_checkpoint_metadata = []
all_checkpoint_embeddings = []


for metadata_path in final_checkpoint_metadata_files:

    embedding_path = (
        metadata_path
        .with_suffix(".npy")
    )

    metadata_chunk = pd.read_parquet(
        metadata_path
    )

    embedding_chunk = np.load(
        embedding_path
    )


    if (
        len(metadata_chunk)
        != embedding_chunk.shape[0]
    ):
        raise ValueError(
            f"Metadata/embedding mismatch "
            f"in {metadata_path.name}"
        )


    all_checkpoint_metadata.append(
        metadata_chunk
    )

    all_checkpoint_embeddings.append(
        embedding_chunk
    )


combined_embedding_metadata = (
    pd.concat(
        all_checkpoint_metadata,
        ignore_index=True
    )
)


combined_embedding_matrix = (
    np.vstack(
        all_checkpoint_embeddings
    )
    .astype(
        np.float32
    )
)


print(
    "Combined metadata rows:",
    len(combined_embedding_metadata)
)

print(
    "Combined embedding shape:",
    combined_embedding_matrix.shape
)

Combined metadata rows: 6681
Combined embedding shape: (6681, 512)


In [77]:
sort_order = np.argsort(
    combined_embedding_metadata[
        "manifest_position"
    ].to_numpy()
)

combined_embedding_metadata = (
    combined_embedding_metadata
    .iloc[
        sort_order
    ]
    .reset_index(drop=True)
)

combined_embedding_matrix = (
    combined_embedding_matrix[
        sort_order
    ]
)

In [78]:
final_embedding_norms = (
    np.linalg.norm(
        combined_embedding_matrix,
        axis=1
    )
)


final_embedding_check = pd.Series({
    "Expected manifest rows":
        len(production_manifest),

    "Embedding metadata rows":
        len(
            combined_embedding_metadata
        ),

    "Embedding matrix rows":
        combined_embedding_matrix.shape[0],

    "Embedding dimension":
        combined_embedding_matrix.shape[1],

    "Unique embedded photo IDs":
        combined_embedding_metadata[
            "photo_id"
        ].nunique(),

    "Duplicate embedded photo IDs":
        combined_embedding_metadata[
            "photo_id"
        ].duplicated().sum(),

    "Photo IDs exactly match final manifest":
        (
            set(
                combined_embedding_metadata[
                    "photo_id"
                ]
            )
            ==
            set(
                production_manifest[
                    "photo_id"
                ]
            )
        ),

    "Photo ID order matches final manifest":
        (
            combined_embedding_metadata[
                "photo_id"
            ].tolist()
            ==
            production_manifest[
                "photo_id"
            ].tolist()
        ),

    "Contains NaN":
        np.isnan(
            combined_embedding_matrix
        ).any(),

    "Contains infinity":
        np.isinf(
            combined_embedding_matrix
        ).any(),

    "Minimum embedding norm":
        float(
            final_embedding_norms.min()
        ),

    "Mean embedding norm":
        float(
            final_embedding_norms.mean()
        ),

    "Maximum embedding norm":
        float(
            final_embedding_norms.max()
        ),

    "All embeddings normalised":
        np.allclose(
            final_embedding_norms,
            1.0,
            atol=1e-5
        )
})

final_embedding_check

Expected manifest rows                     6681
Embedding metadata rows                    6681
Embedding matrix rows                      6681
Embedding dimension                         512
Unique embedded photo IDs                  6681
Duplicate embedded photo IDs                  0
Photo IDs exactly match final manifest     True
Photo ID order matches final manifest      True
Contains NaN                              False
Contains infinity                         False
Minimum embedding norm                      1.0
Mean embedding norm                         1.0
Maximum embedding norm                      1.0
All embeddings normalised                  True
dtype: object

In [79]:
final_embedding_matrix_path = (
    clip_output_dir
    / "new_orleans_clip_vit_b32_image_embeddings.npy"
)

final_embedding_index_path = (
    clip_output_dir
    / "new_orleans_clip_vit_b32_image_embedding_index.parquet"
)


# Save final matrix atomically
temporary_final_matrix_path = (
    clip_output_dir
    / "final_embeddings.tmp"
)

with open(
    temporary_final_matrix_path,
    "wb"
) as file_handle:

    np.save(
        file_handle,
        combined_embedding_matrix
    )

temporary_final_matrix_path.replace(
    final_embedding_matrix_path
)


# Save aligned metadata
combined_embedding_metadata.to_parquet(
    final_embedding_index_path,
    index=False,
    engine="pyarrow"
)


print(
    "Final embedding matrix:"
)

print(
    final_embedding_matrix_path.resolve()
)

print(
    "\nFinal embedding index:"
)

print(
    final_embedding_index_path.resolve()
)

Final embedding matrix:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/clip_vit_b32_embeddings/new_orleans_clip_vit_b32_image_embeddings.npy

Final embedding index:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/clip_vit_b32_embeddings/new_orleans_clip_vit_b32_image_embedding_index.parquet


## Stage D5.1 — Aggregate Image Embeddings to Business Level

Image-level CLIP embeddings are aggregated into a single visual representation
for each image-supported business.

A label-balanced mean pooling strategy is used. Images belonging to the same
Yelp photo label are first averaged to produce a label-level representation.
The available label representations for each business are then averaged and
L2-normalised.

This prevents businesses with multiple photographs from the same visual
category from disproportionately weighting that category in the final business
representation.

In [80]:
# --------------------------------------------------
# Load final image-level CLIP outputs
# --------------------------------------------------

image_embedding_matrix = np.load(
    final_embedding_matrix_path
)

image_embedding_index = (
    pd.read_parquet(
        final_embedding_index_path
    )
    .reset_index(drop=True)
)


print(
    "Image embedding matrix:",
    image_embedding_matrix.shape
)

print(
    "Image index rows:",
    len(image_embedding_index)
)

print(
    "Businesses represented:",
    image_embedding_index[
        "business_id"
    ].nunique()
)

Image embedding matrix: (6681, 512)
Image index rows: 6681
Businesses represented: 1977


In [81]:
image_level_aggregation_input_check = pd.Series({
    "Embedding rows":
        image_embedding_matrix.shape[0],

    "Index rows":
        len(image_embedding_index),

    "Embedding dimension":
        image_embedding_matrix.shape[1],

    "Unique photo IDs":
        image_embedding_index[
            "photo_id"
        ].nunique(),

    "Unique businesses":
        image_embedding_index[
            "business_id"
        ].nunique(),

    "Duplicate photo IDs":
        image_embedding_index[
            "photo_id"
        ].duplicated().sum(),

    "Contains NaN":
        np.isnan(
            image_embedding_matrix
        ).any(),

    "Contains infinity":
        np.isinf(
            image_embedding_matrix
        ).any()
})

image_level_aggregation_input_check

Embedding rows          6681
Index rows              6681
Embedding dimension      512
Unique photo IDs        6681
Unique businesses       1977
Duplicate photo IDs        0
Contains NaN           False
Contains infinity      False
dtype: object

In [82]:
image_embedding_index[
    "embedding_row"
] = np.arange(
    len(image_embedding_index)
)

image_embedding_index[
    "label"
] = (
    image_embedding_index[
        "label"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [83]:
business_embedding_records = []
business_embedding_vectors = []


for business_id, business_rows in (
    image_embedding_index.groupby(
        "business_id",
        sort=True
    )
):

    # ----------------------------------------------
    # First level:
    # average images within each visual label
    # ----------------------------------------------

    label_vectors = []

    for label, label_rows in (
        business_rows.groupby(
            "label",
            sort=True
        )
    ):

        row_positions = (
            label_rows[
                "embedding_row"
            ]
            .to_numpy()
        )

        label_image_vectors = (
            image_embedding_matrix[
                row_positions
            ]
        )

        label_mean_vector = (
            label_image_vectors.mean(
                axis=0
            )
        )

        label_vectors.append(
            label_mean_vector
        )


    # ----------------------------------------------
    # Second level:
    # equally average available visual labels
    # ----------------------------------------------

    business_vector = (
        np.vstack(
            label_vectors
        )
        .mean(
            axis=0
        )
    )


    # ----------------------------------------------
    # L2-normalise final business representation
    # ----------------------------------------------

    business_norm = np.linalg.norm(
        business_vector
    )

    if business_norm <= 0:
        raise ValueError(
            f"Zero-norm business embedding "
            f"detected for {business_id}"
        )

    business_vector = (
        business_vector
        / business_norm
    ).astype(
        np.float32
    )


    # ----------------------------------------------
    # Store vector + aligned metadata
    # ----------------------------------------------

    business_embedding_vectors.append(
        business_vector
    )

    business_embedding_records.append({
        "business_id":
            business_id,

        "image_count":
            len(
                business_rows
            ),

        "unique_image_labels":
            business_rows[
                "label"
            ].nunique(),

        "image_labels":
            "|".join(
                sorted(
                    business_rows[
                        "label"
                    ].unique()
                )
            ),

        "in_personalisation_core":
            bool(
                business_rows[
                    "in_personalisation_core"
                ].iloc[0]
            ),

        "aggregation_method":
            "label_balanced_mean"
    })

In [84]:
business_image_embedding_matrix = (
    np.vstack(
        business_embedding_vectors
    )
    .astype(
        np.float32
    )
)

business_image_embedding_index = (
    pd.DataFrame(
        business_embedding_records
    )
)


print(
    "Business embedding matrix:",
    business_image_embedding_matrix.shape
)

print(
    "Business index rows:",
    len(
        business_image_embedding_index
    )
)

Business embedding matrix: (1977, 512)
Business index rows: 1977


In [85]:
business_image_embedding_index[
    [
        "image_count",
        "unique_image_labels"
    ]
].describe()

,image_count,unique_image_labels
count,1977.000000,1977.000000
mean,3.379363,2.065756
std,1.665541,1.083426
min,1.000000,1.000000
25%,2.000000,1.000000
50%,4.000000,2.000000
75%,5.000000,3.000000
max,5.000000,5.000000


In [86]:
business_embedding_norms = (
    np.linalg.norm(
        business_image_embedding_matrix,
        axis=1
    )
)


business_embedding_check = pd.Series({
    "Expected businesses":
        image_embedding_index[
            "business_id"
        ].nunique(),

    "Business index rows":
        len(
            business_image_embedding_index
        ),

    "Business embedding rows":
        business_image_embedding_matrix.shape[0],

    "Embedding dimension":
        business_image_embedding_matrix.shape[1],

    "Unique business IDs":
        business_image_embedding_index[
            "business_id"
        ].nunique(),

    "Duplicate business IDs":
        business_image_embedding_index[
            "business_id"
        ].duplicated().sum(),

    "Total source images represented":
        business_image_embedding_index[
            "image_count"
        ].sum(),

    "Maximum images for one business":
        business_image_embedding_index[
            "image_count"
        ].max(),

    "Minimum images for one business":
        business_image_embedding_index[
            "image_count"
        ].min(),

    "Maximum unique labels":
        business_image_embedding_index[
            "unique_image_labels"
        ].max(),

    "Personalisation businesses":
        business_image_embedding_index.loc[
            business_image_embedding_index[
                "in_personalisation_core"
            ],
            "business_id"
        ].nunique(),

    "General-only businesses":
        business_image_embedding_index.loc[
            ~business_image_embedding_index[
                "in_personalisation_core"
            ],
            "business_id"
        ].nunique(),

    "Contains NaN":
        np.isnan(
            business_image_embedding_matrix
        ).any(),

    "Contains infinity":
        np.isinf(
            business_image_embedding_matrix
        ).any(),

    "Minimum norm":
        float(
            business_embedding_norms.min()
        ),

    "Mean norm":
        float(
            business_embedding_norms.mean()
        ),

    "Maximum norm":
        float(
            business_embedding_norms.max()
        ),

    "All embeddings normalised":
        np.allclose(
            business_embedding_norms,
            1.0,
            atol=1e-5
        )
})

business_embedding_check

Expected businesses                 1977
Business index rows                 1977
Business embedding rows             1977
Embedding dimension                  512
Unique business IDs                 1977
Duplicate business IDs                 0
Total source images represented     6681
Maximum images for one business        5
Minimum images for one business        1
Maximum unique labels                  5
Personalisation businesses          1729
General-only businesses              248
Contains NaN                       False
Contains infinity                  False
Minimum norm                         1.0
Mean norm                            1.0
Maximum norm                         1.0
All embeddings normalised           True
dtype: object

In [87]:
business_embedding_matrix_path = (
    clip_output_dir
    / "new_orleans_clip_vit_b32_business_embeddings.npy"
)

business_embedding_index_path = (
    clip_output_dir
    / "new_orleans_clip_vit_b32_business_embedding_index.parquet"
)


np.save(
    business_embedding_matrix_path,
    business_image_embedding_matrix
)

business_image_embedding_index.to_parquet(
    business_embedding_index_path,
    index=False,
    engine="pyarrow"
)


print(
    "Business embedding matrix saved:"
)

print(
    business_embedding_matrix_path.resolve()
)

print(
    "\nBusiness embedding index saved:"
)

print(
    business_embedding_index_path.resolve()
)

Business embedding matrix saved:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/clip_vit_b32_embeddings/new_orleans_clip_vit_b32_business_embeddings.npy

Business embedding index saved:
/Users/preye/Downloads/Masters AI:ML/Masters Project/Dataset_exploration/processed_data/new_orleans_image_pipeline/clip_vit_b32_embeddings/new_orleans_clip_vit_b32_business_embedding_index.parquet


## Stage D5.2 — Align Visual Features to the Personalisation Cohort

Business-level CLIP representations are aligned to the complete personalisation
business cohort used in the recommendation experiment.

Businesses with genuine visual representations retain their 512-dimensional
CLIP features. Businesses without usable images are retained in the cohort and
receive zero-valued placeholder vectors together with an explicit visual
availability mask.

The mask will later control the multimodal fusion mechanism so that the visual
branch contributes only when genuine visual evidence exists. Visual
availability itself is not introduced as an independent predictive feature.

This preserves an identical 2,516-business population for the non-visual and
multimodal KGRec-inspired configurations.

In [88]:
# --------------------------------------------------
# Load saved business-level visual representations
# --------------------------------------------------

saved_business_visual_matrix = np.load(
    business_embedding_matrix_path
)

saved_business_visual_index = (
    pd.read_parquet(
        business_embedding_index_path
    )
    .reset_index(drop=True)
)

print(
    "Saved business visual matrix:",
    saved_business_visual_matrix.shape
)

print(
    "Saved visual businesses:",
    len(saved_business_visual_index)
)

print(
    "Unique visual businesses:",
    saved_business_visual_index[
        "business_id"
    ].nunique()
)

Saved business visual matrix: (1977, 512)
Saved visual businesses: 1977
Unique visual businesses: 1977


In [89]:
saved_business_visual_index[
    "source_embedding_row"
] = np.arange(
    len(saved_business_visual_index)
)

visual_business_to_row = dict(
    zip(
        saved_business_visual_index[
            "business_id"
        ],
        saved_business_visual_index[
            "source_embedding_row"
        ]
    )
)

print(
    "Visual business mappings:",
    len(visual_business_to_row)
)

Visual business mappings: 1977


In [90]:
personalisation_visual_index = (
    personalisation_businesses[
        ["business_id"]
    ]
    .drop_duplicates()
    .sort_values(
        "business_id"
    )
    .reset_index(drop=True)
)

personalisation_visual_index[
    "visual_embedding_row"
] = np.arange(
    len(personalisation_visual_index)
)

print(
    "Personalisation businesses:",
    len(personalisation_visual_index)
)

print(
    "Unique business IDs:",
    personalisation_visual_index[
        "business_id"
    ].nunique()
)

Personalisation businesses: 2516
Unique business IDs: 2516


In [91]:
VISUAL_EMBEDDING_DIM = 512

personalisation_visual_matrix = np.zeros(
    (
        len(personalisation_visual_index),
        VISUAL_EMBEDDING_DIM
    ),
    dtype=np.float32
)

personalisation_visual_mask = np.zeros(
    len(personalisation_visual_index),
    dtype=bool
)


for target_row, business_id in enumerate(
    personalisation_visual_index[
        "business_id"
    ]
):

    source_row = visual_business_to_row.get(
        business_id
    )

    if source_row is not None:

        personalisation_visual_matrix[
            target_row
        ] = saved_business_visual_matrix[
            source_row
        ]

        personalisation_visual_mask[
            target_row
        ] = True


personalisation_visual_index[
    "has_visual_feature"
] = personalisation_visual_mask


print(
    "Aligned visual matrix:",
    personalisation_visual_matrix.shape
)

print(
    "Businesses with visual features:",
    int(
        personalisation_visual_mask.sum()
    )
)

print(
    "Businesses without visual features:",
    int(
        (~personalisation_visual_mask).sum()
    )
)

Aligned visual matrix: (2516, 512)
Businesses with visual features: 1729
Businesses without visual features: 787


In [92]:
visual_analysis_metadata = (
    saved_business_visual_index[
        [
            "business_id",
            "image_count",
            "unique_image_labels",
            "image_labels",
            "aggregation_method"
        ]
    ]
    .copy()
)


personalisation_visual_index = (
    personalisation_visual_index
    .merge(
        visual_analysis_metadata,
        on="business_id",
        how="left",
        validate="one_to_one"
    )
)


# Businesses without images genuinely have zero visual evidence.
personalisation_visual_index[
    "image_count"
] = (
    personalisation_visual_index[
        "image_count"
    ]
    .fillna(0)
    .astype(int)
)

personalisation_visual_index[
    "unique_image_labels"
] = (
    personalisation_visual_index[
        "unique_image_labels"
    ]
    .fillna(0)
    .astype(int)
)

personalisation_visual_index[
    "image_labels"
] = (
    personalisation_visual_index[
        "image_labels"
    ]
    .fillna("")
)

personalisation_visual_index[
    "aggregation_method"
] = (
    personalisation_visual_index[
        "aggregation_method"
    ]
    .fillna("no_visual_feature")
)

In [93]:
visual_availability_summary = pd.Series({
    "Personalisation businesses":
        len(
            personalisation_visual_index
        ),

    "Businesses with visual feature":
        personalisation_visual_index[
            "has_visual_feature"
        ].sum(),

    "Businesses without visual feature":
        (
            ~personalisation_visual_index[
                "has_visual_feature"
            ]
        ).sum(),

    "Visual coverage percentage":
        round(
            personalisation_visual_index[
                "has_visual_feature"
            ].mean()
            * 100,
            2
        ),

    "Maximum selected images":
        personalisation_visual_index[
            "image_count"
        ].max(),

    "Minimum selected images where visual exists":
        personalisation_visual_index.loc[
            personalisation_visual_index[
                "has_visual_feature"
            ],
            "image_count"
        ].min(),

    "Maximum unique visual labels":
        personalisation_visual_index[
            "unique_image_labels"
        ].max()
})

visual_availability_summary

Personalisation businesses                     2516.00
Businesses with visual feature                 1729.00
Businesses without visual feature               787.00
Visual coverage percentage                       68.72
Maximum selected images                           5.00
Minimum selected images where visual exists       1.00
Maximum unique visual labels                      5.00
dtype: float64

In [94]:
personalisation_visual_norms = np.linalg.norm(
    personalisation_visual_matrix,
    axis=1
)


expected_personalisation_ids = set(
    personalisation_businesses[
        "business_id"
    ]
)

aligned_personalisation_ids = set(
    personalisation_visual_index[
        "business_id"
    ]
)


visual_mask_alignment_check = pd.Series({
    "Expected personalisation businesses":
        len(expected_personalisation_ids),

    "Aligned business rows":
        len(
            personalisation_visual_index
        ),

    "Visual matrix rows":
        personalisation_visual_matrix.shape[0],

    "Visual embedding dimension":
        personalisation_visual_matrix.shape[1],

    "Unique aligned business IDs":
        personalisation_visual_index[
            "business_id"
        ].nunique(),

    "Duplicate business IDs":
        personalisation_visual_index[
            "business_id"
        ].duplicated().sum(),

    "Business IDs exactly match personalisation cohort":
        (
            expected_personalisation_ids
            ==
            aligned_personalisation_ids
        ),

    "Businesses with genuine visual features":
        int(
            personalisation_visual_mask.sum()
        ),

    "Businesses with visual branch masked":
        int(
            (~personalisation_visual_mask).sum()
        ),

    "Mask matches image-count availability":
        np.array_equal(
            personalisation_visual_mask,
            (
                personalisation_visual_index[
                    "image_count"
                ].to_numpy()
                > 0
            )
        ),

    "Available visual vectors normalised":
        np.allclose(
            personalisation_visual_norms[
                personalisation_visual_mask
            ],
            1.0,
            atol=1e-5
        ),

    "Masked visual vectors exactly zero":
        np.all(
            personalisation_visual_matrix[
                ~personalisation_visual_mask
            ]
            == 0
        ),

    "Contains NaN":
        np.isnan(
            personalisation_visual_matrix
        ).any(),

    "Contains infinity":
        np.isinf(
            personalisation_visual_matrix
        ).any()
})

visual_mask_alignment_check

Expected personalisation businesses                   2516
Aligned business rows                                 2516
Visual matrix rows                                    2516
Visual embedding dimension                             512
Unique aligned business IDs                           2516
Duplicate business IDs                                   0
Business IDs exactly match personalisation cohort     True
Businesses with genuine visual features               1729
Businesses with visual branch masked                   787
Mask matches image-count availability                 True
Available visual vectors normalised                   True
Masked visual vectors exactly zero                    True
Contains NaN                                         False
Contains infinity                                    False
dtype: object

In [95]:
personalisation_visual_matrix_path = (
    clip_output_dir
    / "new_orleans_personalisation_visual_features.npy"
)

personalisation_visual_mask_path = (
    clip_output_dir
    / "new_orleans_personalisation_visual_mask.npy"
)

personalisation_visual_index_path = (
    clip_output_dir
    / "new_orleans_personalisation_visual_feature_index.parquet"
)


np.save(
    personalisation_visual_matrix_path,
    personalisation_visual_matrix
)

np.save(
    personalisation_visual_mask_path,
    personalisation_visual_mask
)

personalisation_visual_index.to_parquet(
    personalisation_visual_index_path,
    index=False,
    engine="pyarrow"
)


print(
    "Visual feature matrix saved:",
    personalisation_visual_matrix_path.exists()
)

print(
    "Visual mask saved:",
    personalisation_visual_mask_path.exists()
)

print(
    "Visual index saved:",
    personalisation_visual_index_path.exists()
)

Visual feature matrix saved: True
Visual mask saved: True
Visual index saved: True
